Load csv dataset and drop low minute players

In [39]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

MIN_MINUTES = 900 #10 full matches
players = pd.read_csv("../data/players_data-2025_2026.csv", index_col='Rk')
players = players[players['Min'] >= MIN_MINUTES]

DROP_FEATURES = ['Rk_stats_keeper', 'Nation_stats_keeper', 'Pos_stats_keeper', 'Comp_stats_keeper', 'Age_stats_keeper',
                 'Born_stats_keeper', 'MP_stats_keeper', 'Starts_stats_keeper', 'Min_stats_keeper', '90s_stats_keeper', 
                'PK_stats_shooting', 'PKatt_stats_shooting', 'Rk_stats_playing_time', 'Nation_stats_playing_time', 'Pos_stats_playing_time', 
                'Comp_stats_playing_time', 'Age_stats_playing_time', 'Born_stats_playing_time', 'MP_stats_playing_time', 'Min_stats_playing_time',
                 'Rk_stats_shooting', 'Nation_stats_shooting', 'Pos_stats_shooting', 'Comp_stats_shooting', 'Age_stats_shooting', 'Born_stats_shooting',
                 '90s_stats_shooting', 'Gls_stats_shooting', 'Rk_stats_misc', 'Nation_stats_misc', 'Pos_stats_misc', 'Comp_stats_misc', 'Age_stats_misc', 
                'Born_stats_misc', '90s_stats_misc', 'CrdY_stats_misc', 'CrdR_stats_misc', '90s_stats_playing_time', 'Starts_stats_playing_time']
                 
players = players.drop(columns=DROP_FEATURES)

for col in players.columns:
    print(col, " ", end="")

Player  Nation  Pos  Squad  Comp  Age  Born  MP  Starts  Min  90s  Gls  Ast  G+A  G-PK  PK  PKatt  CrdY  CrdR  G+A-PK  GA  GA90  SoTA  Saves  Save%  W  D  L  CS  CS%  PKatt_stats_keeper  PKA  PKsv  PKm  Sh  SoT  SoT%  Sh/90  SoT/90  G/Sh  G/SoT  Mn/MP  Min%  Mn/Start  Compl  Subs  Mn/Sub  unSub  PPM  onG  onGA  +/-  +/-90  On-Off  2CrdY  Fls  Fld  Off  Crs  Int  TklW  OG  

Attacking Stats -> [Gls: Goals, Ast: Assists, G+A: Goals + Assists, xG: Expected Goals, xAG: Expected Assists, npxG: Non-penalty ExG, G-PK: Non-penalty goals]
Defensive Stats -> [Tkl: Tackles, TklW: Tackles Won, Blocks: Blocks Made, Int: Interceptions, Tkl+Int: Tackles + Interceptions, Clr: Clearances, Err: Errors Leading to Goals]
Passing/Creativity Stats -> [PrgP: Progressive Passes, PrgC: Progressive Carries, KP: Key Passes, Cmp_stats_passing: Pass Completion Percentatage, Ast_stats_passing: Assits, xA: Expected Assists, PPA: Passes into the penalty Area]
Goalkeeping Stats -> [GA: Goals Conceded, Saves: Saves Made, Save%: Save Percentage, CS: Clean Sheets, CS% Clean Sheet Percentage, PKA: Penalites Faces, PKsv: Penalty Saves]
Possession/Ball Control Stats -> [Touches: Total Touches, Carries: Total Carries, PrgRL Progressive Runs, Miscontrols, Dis: Times dispossed]
Extra Stats -> [CrdY: Yellow Cards, CrdR: Red Cards, PKwon: Penalties Won, PKcon: Penalties Conceded, Revoc: Ball Recoveries]

In [40]:
leagues = players['Comp'].unique().to_numpy()
print(leagues)
positions = players['Pos'].unique().to_numpy()
print(positions)

['eng Premier League' 'fr Ligue 1' 'es La Liga' 'it Serie A'
 'de Bundesliga']
['MF,FW' 'MF' 'MF,DF' 'FW,MF' 'DF,MF' 'DF' 'FW' 'GK']


In [86]:
GOAL_KEEPER_FEATURES = ['GA', 'Saves', 'Save%', 'CS', 'CS%', 'PKsv', '90s']
OUTFIELD_FEATURES = ['Gls', 'Ast', 'Sh', 'SoT/90', 'G/Sh', 'G/SoT', 'Fls', 'Fld', 'Off', 'Int','TklW','90s']
QUALITATIVE_FEATURES = ['Player','Pos', 'Nation', 'Squad', 'Comp', 'Born']

In [48]:
prem = players[players['Comp'] == 'eng Premier League']

In [49]:
prem_outfield = prem[prem['Pos'] != 'GK']
prem_outfield = prem[OUTFIELD_FEATURES]

#Missing values found in G/Sh and G/SoT, fill with mean
prem_outfield['G/Sh'] = prem_outfield['G/Sh'].fillna(prem_outfield['G/Sh'].mean())
prem_outfield['G/SoT'] = prem_outfield['G/SoT'].fillna(prem_outfield['G/SoT'].mean())

#Adjusting Stats on a Per 90 Basis
cols_to_change = ['Gls', 'Ast', 'Sh', 'Fls', 'Fld', 'Off', 'Int', 'TklW']
prem_outfield[cols_to_change] = prem_outfield[cols_to_change].apply(lambda x: x / prem_outfield['90s'])
prem_outfield = prem_outfield.rename(columns={'Gls': 'Gls/90', 'Ast': 'Ast/90', 'Sh': 'Sh/90', 
                                              'Fls': 'Fls/90', 'Fld': 'Fld/90', 'Off': 'Off/90', 
                                              'Int': 'Int/90', 'TklW': 'TklW/90'})
prem_outfield = prem_outfield.drop(columns='90s')
prem_outfield.head()

,Gls/90,Ast/90,Sh/90,SoT/90,G/Sh,G/SoT,Fls/90,Fld/90,Off/90,Int/90,TklW/90
Rk,,,,,,,,,,,
1,0.147059,0.183824,1.727941,0.62,0.09,0.24,0.735294,1.875000,0.183824,0.625000,0.992647
25,0.101523,0.101523,0.507614,0.30,0.20,0.33,1.827411,0.659898,0.050761,1.522843,1.218274
34,0.247934,0.082645,1.652893,0.74,0.15,0.33,2.396694,1.900826,0.413223,0.495868,1.404959
40,0.000000,0.000000,0.819672,0.25,0.00,0.00,0.819672,0.409836,0.081967,1.147541,1.721311
55,0.000000,0.000000,0.681818,0.06,0.00,0.00,0.738636,0.852273,0.000000,1.477273,1.193182


In [94]:
ADJUSTED_FEATURES = ['Gls/90', 'Ast/90', 'Sh/90', 'Fls/90', 'Fld/90', 'Off/90', 'Int/90', 'TklW/90',]

scaler = StandardScaler()

prem_outfield[ADJUSTED_FEATURES] = scaler.fit_transform(prem_outfield[ADJUSTED_FEATURES])
prem_outfield

,Gls/90,Ast/90,Sh/90,SoT/90,G/Sh,G/SoT,Fls/90,Fld/90,Off/90,Int/90,TklW/90
Rk,,,,,,,,,,,
1,0.165347,1.088148,0.654113,0.62,0.09,0.24,-0.527613,1.492903,0.212143,-0.224866,0.138402
25,-0.142570,0.185288,-0.762689,0.30,0.20,0.33,1.561762,-0.501346,-0.491235,1.734714,0.565402
34,0.847471,-0.021811,0.566981,0.74,0.15,0.33,2.650882,1.535290,1.424768,-0.506702,0.918704
40,-0.829074,-0.928444,-0.400389,0.25,0.00,0.00,-0.366186,-0.911754,-0.326279,0.915602,1.517403
55,-0.829074,-0.928444,-0.560438,0.06,0.00,0.00,-0.521219,-0.185618,-0.759564,1.635256,0.517915
...,...,...,...,...,...,...,...,...,...,...,...
2783,-0.619722,1.109375,-0.597199,0.15,0.05,0.20,-0.572036,0.448086,-0.759564,0.370604,0.193331
2787,-0.599851,-0.184698,-0.722334,0.14,0.06,0.25,0.465199,-0.916770,-0.580375,0.482611,1.210837
2794,-0.829074,-0.928444,-0.810630,0.16,0.00,0.00,-0.249184,-1.159199,-0.211783,1.916684,0.220958


Outfield players sorted, now for Goalkeepers

In [106]:
prem_keepers = prem[prem['Pos'] == 'GK']
prem_keepers = prem_keepers[GOAL_KEEPER_FEATURES]
#found no missing values

cols_to_change = ['GA', 'Saves', 'CS', 'PKsv']
prem_keepers[cols_to_change] = prem_keepers[cols_to_change].apply(lambda x : x / prem_keepers['90s'])

prem_keepers = prem_keepers.rename(columns={'GA': 'GA/90', 'Saves': 'Saves/90', 'CS':'CS/90', 'PKsv': 'PKsv/90'})
prem_keepers = prem_keepers.drop(columns='90s')

ADJUSTED_COLS = ['GA/90', 'Saves/90', 'CS/90', 'PKsv/90']
prem_keepers[ADJUSTED_COLS] = scaler.fit_transform(prem_keepers[ADJUSTED_COLS])

prem_keepers.head()

,GA/90,Saves/90,Save%,CS/90,CS%,PKsv/90
Rk,,,,,,
84,-0.722379,-1.237309,66.3,0.550248,30.8,-0.606464
143,1.431302,2.132114,70.4,-2.131615,0.0,-0.606464
639,-0.607882,0.169671,71.6,-0.150693,22.7,-0.606464
753,-1.833669,-1.208859,72.4,1.713703,44.1,0.899562
773,2.016052,1.772832,66.1,-1.135494,11.4,-0.606464


Now for LaLiga

In [107]:
laliga = players[players['Comp'] == 'es La Liga']

laliga.head()

,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,...,+/-90,On-Off,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG
Rk,,,,,,,,,,,,,,,,,,,,,
15,Abdel Abqar,ma MAR,DF,Getafe,es La Liga,26.0,1999.0,23,17,1437,...,0.06,0.38,0,30,36,1,2,20,17,0
23,Akor Adams,ng NGA,FW,Sevilla,es La Liga,25.0,2000.0,32,21,2064,...,-0.35,0.05,0,36,12,41,17,1,4,0
38,David Affengruber,at AUT,DF,Elche,es La Liga,24.0,2001.0,36,33,2949,...,-0.12,0.64,0,34,35,2,8,50,50,0
41,Julen Agirrezabala,es ESP,GK,Valencia,es La Liga,24.0,2000.0,18,18,1620,...,-0.72,-0.92,0,0,1,0,0,0,1,0
42,Lucien Agoume,fr FRA,MF,Sevilla,es La Liga,23.0,2002.0,34,31,2686,...,-0.30,0.31,0,58,16,8,18,49,40,0


In [124]:
laliga_outfield = laliga[OUTFIELD_FEATURES]

#missing features in G/Sh and G/SoT, fill with mean
laliga_outfield['G/Sh'] = laliga_outfield['G/Sh'].fillna(laliga_outfield['G/Sh'].mean())
laliga_outfield['G/SoT'] = laliga_outfield['G/SoT'].fillna(laliga_outfield['G/SoT'].mean())

cols_to_change = ['Gls', 'Ast', 'Sh', 'Fls', 'Fld', 'Off', 'Int', 'TklW']
laliga_outfield[cols_to_change] = laliga_outfield[cols_to_change].apply(lambda x : x / laliga_outfield['90s'])

laliga_outfield = laliga_outfield.rename(columns={'Gls': 'Gls/90', 'Ast': 'Ast/90', 'Sh': 'Sh/90', 
                                              'Fls': 'Fls/90', 'Fld': 'Fld/90', 'Off': 'Off/90', 
                                              'Int': 'Int/90', 'TklW': 'TklW/90'})
laliga_outfield = laliga_outfield.drop(columns='90s')

ADJUSTED_FEATURES = ['Gls/90', 'Ast/90', 'Sh/90', 'Fls/90', 'Fld/90', 'Off/90', 'Int/90', 'TklW/90']

laliga_outfield[ADJUSTED_FEATURES] = scaler.fit_transform(laliga_outfield[ADJUSTED_FEATURES])

laliga_outfield.head()

,Gls/90,Ast/90,Sh/90,SoT/90,G/Sh,G/SoT,Fls/90,Fld/90,Off/90,Int/90,TklW/90
Rk,,,,,,,,,,,
15,-0.760052,0.378415,-0.769907,0.06,0.000000,0.00000,1.306971,1.563255,-0.446649,1.096545,0.286804
23,1.816479,0.441965,1.543027,1.31,0.110000,0.23000,0.757200,-0.837800,6.047125,-1.521552,-1.531221
38,-0.580166,-0.621892,-0.780993,0.12,0.070000,0.25000,-0.214534,-0.082343,-0.452378,1.692053,1.232628
41,-0.760052,-0.944572,-1.224441,0.00,0.084123,0.27202,-2.095667,-1.489488,-0.681536,-1.616324,-1.775140
42,-0.562057,0.120921,-0.701487,0.07,0.070000,0.50000,1.436378,-0.819861,0.327377,1.952283,0.859720


In [130]:
laliga_keepers = laliga[laliga['Pos'] == 'GK']
laliga_keepers = laliga_keepers[GOAL_KEEPER_FEATURES]
#found no missing values

cols_to_change = ['GA', 'Saves', 'CS', 'PKsv']
laliga_keepers[cols_to_change] = laliga_keepers[cols_to_change].apply(lambda x : x / laliga_keepers['90s'])

laliga_keepers = laliga_keepers.rename(columns={'GA': 'GA/90', 'Saves': 'Saves/90', 'CS':'CS/90', 'PKsv': 'PKsv/90'})
laliga_keepers = laliga_keepers.drop(columns='90s')

ADJUSTED_COLS = ['GA/90', 'Saves/90', 'CS/90', 'PKsv/90']
laliga_keepers[ADJUSTED_COLS] = scaler.fit_transform(laliga_keepers[ADJUSTED_COLS])

laliga_keepers.head()

,GA/90,Saves/90,Save%,CS/90,CS%,PKsv/90
Rk,,,,,,
41,1.232244,-0.089289,65.8,-0.142401,22.2,1.592642
240,-0.934690,-0.596391,73.8,0.802792,32.4,0.422079
601,-1.733517,-1.763701,74.5,1.574571,40.6,-0.894804
719,-0.328683,-0.778948,70.3,0.116765,25.0,-0.894804
734,-0.073258,-0.041061,71.9,-0.519369,18.2,-0.894804
